TEST CASES G-EVALS

In [2]:
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What's your refund policy?",
        actual_output=(
            "Happy to help! We offer full refunds within 30 days of purchase, no questions "
            "asked. Just reply here with your order number."
        ),
        expected_output="Refunds are available within 30 days of purchase.",
    ),
    LLMTestCase(
        input="What's your refund policy?",
        actual_output="No refunds. Read the policy page next time.",
        expected_output="Refunds are available within 30 days of purchase.",
    ),
]

d:\Gen AI Projects\DeepEvals\.venv\Lib\site-packages\deepeval\evaluate\execute\loop.py:809: SyntaxWarning: 'return' in a 'finally' block
  return


THE JUDGE : LOCALMODEL (GROQ)

In [3]:
from deepeval.models import LocalModel
import os
judge = LocalModel(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)
print("Judge ready:", judge.get_model_name())

Judge ready: llama-3.3-70b-versatile (Local Model)


DEFINE TWO G-EVAL METRICS

In [4]:
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# correctness = GEval(
#     name="Correctness",
#     criteria="Determine whether the actual output is factually correct given the expected output.",
#     evaluation_params=[
#         SingleTurnParams.INPUT,
#         SingleTurnParams.ACTUAL_OUTPUT,
#         SingleTurnParams.EXPECTED_OUTPUT,
#     ],
#     model=judge,
# )

tone = GEval(
    name="Professional Tone",
    evaluation_steps=[
        "Check whether the response is polite and professional.",
        "Penalize sarcasm, rudeness, or dismissive language.",
        "Reward clear, respectful phrasing even if brief.",
    ],
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT],
    model=judge,
)

In [5]:
from deepeval.metrics.g_eval import Rubric, GEval
from deepeval.test_case import SingleTurnParams

correctness = GEval(
    name="Correctness",
    criteria="Determine whether the actual output is factually correct given the expected output.",
    evaluation_params=[
        SingleTurnParams.INPUT,
        SingleTurnParams.ACTUAL_OUTPUT,
        SingleTurnParams.EXPECTED_OUTPUT,
    ],
    rubric=[
        Rubric(score_range=(0, 2), expected_outcome="Factually wrong or contradicts the expected output."),
        Rubric(score_range=(3, 5), expected_outcome="Partially correct -- misses or garbles a key fact."),
        Rubric(score_range=(6, 8), expected_outcome="Mostly correct with only minor omissions."),
        Rubric(score_range=(9, 10), expected_outcome="Fully correct and complete."),
    ],
    model=judge,  # your existing LocalModel(Groq) judge
)

RUN THE EVAL PIPELINE

In [ ]:
from deepeval import evaluate
from deepeval.evaluate.configs import AsyncConfig, DisplayConfig, ErrorConfig

results = evaluate(
    test_cases=test_cases,
    metrics=[correctness, tone],
    async_config=AsyncConfig(max_concurrent=2),           # be gentle on Groq's rate limits
    display_config=DisplayConfig(print_results=True),
    error_config=ErrorConfig(ignore_errors=True, skip_on_missing_params=True),## also write false
)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using llama-3.3-70b-versatile (Local Model), 
strict=False, async_mode=True)...

✨ You're running DeepEval's latest Professional Tone [GEval] Metric! (using llama-3.3-70b-versatile (Local Model),
strict=False, async_mode=True)...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

d:\Gen AI Projects\DeepEvals\.venv\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

In get_cached_test_run, temp=False, Lock acquisition failed: Shared locks on Windows require the win32 extra 
(pywin32); msvcrt provides no true shared lock. Install it with: pip install "portalocker[win32]"

...

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

Warning: Could not load test run from disk: Shared locks on Windows require the win32 extra (pywin32); msvcrt 
provides no true shared lock. Install it with: pip install "portalocker"

⚠ WARNING: All metrics errored across every test case — no metric scores were recorded. Posting the run anyway so 
you can inspect the trace + span errors on the Confident AI dashboard.

⚠ WARNING: No hyperparameters logged.
» ]8;id=15680900;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.9s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 2

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.